# Phase 3: Batch Feature Engineering & Detection Flags

## Overview
This notebook transforms event-level data from Phase 2 into file-level features with logic-based detection flags for Phase 4 model training.

## Detection Logic Scope (Oh et al.)
**Core Logic:**
- LogFile-1A (Algorithms 1-4): Timestamp change extraction and validation
- UsnJrnl-1A (Algorithms 5-7): BASIC_INFO_CHANGE pattern detection

**Additional Validation:**
- LogFile-4 (Algorithm 10): $FN timestamp manipulation via file move
- UsnJrnl-3 (Algorithm 11): $FN manipulation via USN patterns

## Dataset Split
**Training Datasets (22):** Combined into single feature file for model training
**Validation Datasets (5):** Separate feature files for independent evaluation

## Input
- Phase 2 outputs: `grouped_events_[dataset].csv`

## Output
- File features: `file_features_[dataset].csv`
- Detection flags: `detection_flags_[dataset].csv`
- Combined training: `file_features_training.csv`, `detection_flags_training.csv`


In [13]:
# [Cell 2] Imports and Global Configuration

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# DIRECTORY CONFIGURATION
# =============================================================================

# Phase 2 input directory
PHASE2_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 2: Data Preprocessing & Event Grouping")

# Phase 3 output directory
OUTPUT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# DATASET DEFINITIONS
# =============================================================================

# Training datasets (22 total)
TRAINING_PE = [f"{i:02d}-PE" for i in range(1, 13)]  # 01-PE to 12-PE

TRAINING_APT = [
    "01-APT17", "03-APT21", "04-APT28", "05-APT29", "06-APT30",
    "07-APT37", "08-APT38", "10-DarkHotel663", "11-DarkHotelbbd", "14-Winnti43b"
]

# Validation datasets (5 total)
VALIDATION_APT = ["02-APT19", "09-APT40", "12-Kimsuky", "13-Winnti731"]
VALIDATION_LONEWOLF = ["LoneWolf"]

# Combined lists
ALL_TRAINING = TRAINING_PE + TRAINING_APT
ALL_VALIDATION = VALIDATION_APT + VALIDATION_LONEWOLF

print(f"Training datasets: {len(ALL_TRAINING)}")
print(f"Validation datasets: {len(ALL_VALIDATION)}")
print(f"Total datasets: {len(ALL_TRAINING) + len(ALL_VALIDATION)}")
print(f"\nPhase 2 input: {PHASE2_DIR}")
print(f"Phase 3 output: {OUTPUT_DIR}")


Training datasets: 22
Validation datasets: 5
Total datasets: 27

Phase 2 input: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 2: Data Preprocessing & Event Grouping
Phase 3 output: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling


## Core Processing Functions

All preprocessing functions are defined below. Run this cell once before processing any datasets.

### Functions Overview
1. `get_input_paths()` - Resolve input file paths for a dataset
2. `load_phase1_data()` - Load MFT, LogFile, UsnJrnl CSVs
3. `preprocess_mft()` - Create MFT reference table
4. `process_logfile_events()` - Process and join LogFile events
5. `process_usnjrnl_events()` - Process and join UsnJrnl events
6. `normalize_logfile_schema()` - Normalize LogFile to unified schema
7. `normalize_usnjrnl_schema()` - Normalize UsnJrnl to unified schema
8. `combine_and_sort_events()` - Merge and order event streams
9. `process_dataset()` - Main function to process a single dataset


## Feature Engineering Categories

### A. Timestamp Change Features (LogFile-1A, Algorithm 3)
- `num_timestamp_changes`: Count of IsTimestampChange=True events
- `num_backward_jumps`: Count where Redo < Undo (timestamp moved to past)
- `num_forward_jumps`: Count where Redo > Undo
- `num_creation_changes`: Count where $SI-C changed
- `max_backward_jump_seconds`: Largest backward time delta
- `mean_jump_seconds`: Average absolute time delta
- `timestamp_change_density`: Changes per total events
- `num_zero_nanosecond_events`: Events with zero nanoseconds in Redo timestamps

### B. Structural NTFS Patterns (Algorithm 1-2)
- `only_SI_modified`: No $FN changes detected
- `num_update_resident_value`: Count of UpdateResidentValue operations
- `repeated_update_resident_value`: Multiple UpdateResidentValue ops
- `consecutive_timestamp_changes`: Sequential changes without gaps
- `num_logfile_events`: Total LogFile events for file

### C. Cross-Artifact Consistency (Algorithms 5-7)
- `has_logfile_ts_change`: Any LogFile timestamp change
- `has_usn_basic_info`: Any BASIC_INFO_CHANGE
- `has_usn_close`: Any CLOSE event
- `has_usn_file_create`: Any FILE_CREATE event
- `logfile_usn_mismatch`: Timestamp change without USN evidence
- `has_usn_basic_pattern`: BASIC_INFO_CHANGE + CLOSE pattern

### D. Temporal Behavior Features
- `min_inter_event_delta`: Shortest time between events
- `max_inter_event_delta`: Longest time between events
- `mean_inter_event_delta`: Average time between events
- `burstiness_score`: Concentration in short windows
- `event_time_span_seconds`: Total time span of events


In [14]:
# [Cell 4] Timestamp Analysis Helper Functions
# Reference: Oh et al. Algorithm 3 - Checking Timestamp Changes

def compute_timestamp_jump(undo_ts, redo_ts):
    """
    Compute time delta between Undo (before) and Redo (after) timestamps.
    Returns delta in seconds (negative = backward jump).
    """
    if pd.isna(undo_ts) or pd.isna(redo_ts):
        return np.nan
    return (redo_ts - undo_ts).total_seconds()


def has_zero_nanoseconds(timestamp):
    """
    Check if timestamp has zero nanoseconds (microseconds in pandas).
    Many timestomping tools produce zero nanoseconds.
    """
    if pd.isna(timestamp):
        return False
    return timestamp.microsecond == 0


def is_backward_jump(undo_ts, redo_ts):
    """
    Detect backward timestamp jump (moved to past).
    Algorithm 3: event.before_$SI-M > event.after_$SI-M
    """
    if pd.isna(undo_ts) or pd.isna(redo_ts):
        return False
    return redo_ts < undo_ts


def is_forward_jump(undo_ts, redo_ts):
    """
    Detect forward timestamp jump (moved to future).
    """
    if pd.isna(undo_ts) or pd.isna(redo_ts):
        return False
    return redo_ts > undo_ts


print("Timestamp analysis helper functions defined.")


Timestamp analysis helper functions defined.


In [15]:
# [Cell 5] Per-Event Indicator Computation

def compute_event_indicators(df):
    """
    Compute per-event indicators for feature aggregation.
    
    Parameters:
        df: Grouped events DataFrame from Phase 2
    
    Returns:
        DataFrame with additional indicator columns
    """
    df = df.copy()
    
    # Parse timestamp columns
    ts_cols = ['EventTimestamp', 'Undo_$SI-C', 'Undo_$SI-M', 'Undo_$SI-E', 'Undo_$SI-A',
               'Redo_$SI-C', 'Redo_$SI-M', 'Redo_$SI-E', 'Redo_$SI-A']
    for col in ts_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Event source flags
    df['is_logfile_event'] = df['EventSource'] == 'LogFile'
    df['is_usnjrnl_event'] = df['EventSource'] == 'UsnJrnl'
    
    # Ensure IsTimestampChange is boolean
    df['IsTimestampChange'] = df['IsTimestampChange'].fillna(False).astype(bool)
    
    # Backward/forward jumps using $SI-M (Modified time)
    df['backward_jump_M'] = df.apply(
        lambda row: is_backward_jump(row['Undo_$SI-M'], row['Redo_$SI-M'])
        if row['IsTimestampChange'] else False, axis=1
    )
    
    df['forward_jump_M'] = df.apply(
        lambda row: is_forward_jump(row['Undo_$SI-M'], row['Redo_$SI-M'])
        if row['IsTimestampChange'] else False, axis=1
    )
    
    # Jump magnitude in seconds
    df['jump_seconds_M'] = df.apply(
        lambda row: compute_timestamp_jump(row['Undo_$SI-M'], row['Redo_$SI-M'])
        if row['IsTimestampChange'] else np.nan, axis=1
    )
    
    # Creation time changes (Algorithm 3)
    df['creation_changed'] = df.apply(
        lambda row: (pd.notna(row['Undo_$SI-C']) and 
                     pd.notna(row['Redo_$SI-C']) and 
                     row['Undo_$SI-C'] != row['Redo_$SI-C'])
        if row['IsTimestampChange'] else False, axis=1
    )
    
    # Zero nanoseconds detection
    df['redo_zero_ns_C'] = df['Redo_$SI-C'].apply(has_zero_nanoseconds)
    df['redo_zero_ns_M'] = df['Redo_$SI-M'].apply(has_zero_nanoseconds)
    df['redo_zero_ns_E'] = df['Redo_$SI-E'].apply(has_zero_nanoseconds)
    df['redo_zero_ns_A'] = df['Redo_$SI-A'].apply(has_zero_nanoseconds)
    
    df['has_redo_zero_ns'] = (df['redo_zero_ns_C'] | df['redo_zero_ns_M'] | 
                              df['redo_zero_ns_E'] | df['redo_zero_ns_A'])
    
    return df


print("Per-event indicator computation function defined.")


Per-event indicator computation function defined.


In [16]:
# [Cell 6] Timestamp Change Feature Aggregation (Category A)
# Reference: Oh et al. Algorithm 3

def aggregate_timestamp_features(group):
    """
    Aggregate timestamp change features for a single file.
    """
    ts_events = group[group['IsTimestampChange'] == True]
    total_events = len(group)
    
    # A.1 Count of timestamp changes
    num_ts_changes = len(ts_events)
    
    # A.2 Backward jumps (Algorithm 3: before > after)
    num_backward = group['backward_jump_M'].sum()
    
    # A.3 Forward jumps
    num_forward = group['forward_jump_M'].sum()
    
    # A.4 Creation time changes
    num_creation = group['creation_changed'].sum()
    
    # A.5 Max backward jump magnitude
    backward_jumps = group.loc[group['backward_jump_M'], 'jump_seconds_M']
    max_backward = abs(backward_jumps.min()) if len(backward_jumps) > 0 else 0
    
    # A.6 Mean jump magnitude
    all_jumps = group['jump_seconds_M'].dropna()
    mean_jump = all_jumps.abs().mean() if len(all_jumps) > 0 else 0
    
    # A.7 Timestamp change density
    density = num_ts_changes / total_events if total_events > 0 else 0
    
    # A.8 Zero nanoseconds count
    num_zero_ns = group['has_redo_zero_ns'].sum()
    
    return pd.Series({
        'num_timestamp_changes': num_ts_changes,
        'num_backward_jumps': num_backward,
        'num_forward_jumps': num_forward,
        'num_creation_changes': num_creation,
        'max_backward_jump_seconds': max_backward,
        'mean_jump_seconds': mean_jump,
        'timestamp_change_density': density,
        'num_zero_nanosecond_events': num_zero_ns
    })


print("Timestamp feature aggregation function defined.")


Timestamp feature aggregation function defined.


In [17]:
# [Cell 7] Structural NTFS Feature Aggregation (Category B)
# Reference: Oh et al. Algorithms 1-2

def aggregate_structural_features(group):
    """
    Aggregate structural NTFS pattern features.
    """
    logfile_events = group[group['is_logfile_event']]
    ts_events = group[group['IsTimestampChange'] == True]
    
    # B.1 Only $SI modified (attribute offset 0x18-0x30 = $SI timestamps)
    if len(logfile_events) > 0:
        attr_offsets = logfile_events['AttributeOffset'].dropna()
        only_si = attr_offsets.isin([24, 32, 40, 48]).all() if len(attr_offsets) > 0 else False
    else:
        only_si = False
    
    # B.2 UpdateResidentValue operations (Algorithm 1: redo.op == 0x7)
    if 'RedoOPName' in logfile_events.columns:
        urv_count = (logfile_events['RedoOPName'] == 'UpdateResidentValue').sum()
    else:
        urv_count = 0
    
    # B.3 Repeated UpdateResidentValue
    repeated_urv = urv_count > 1
    
    # B.4 Consecutive timestamp changes
    consecutive = False
    if len(ts_events) >= 2:
        ts_lsns = ts_events['LSN'].dropna().sort_values()
        if len(ts_lsns) >= 2:
            lsn_diffs = ts_lsns.diff().dropna()
            consecutive = (lsn_diffs < 1000).any()
    
    # B.5 Total LogFile events
    num_logfile = len(logfile_events)
    
    return pd.Series({
        'only_SI_modified': only_si,
        'num_update_resident_value': urv_count,
        'repeated_update_resident_value': repeated_urv,
        'consecutive_timestamp_changes': consecutive,
        'num_logfile_events': num_logfile
    })


print("Structural feature aggregation function defined.")


Structural feature aggregation function defined.


In [18]:
# [Cell 8] Cross-Artifact Feature Aggregation (Category C)
# Reference: Oh et al. Algorithms 5-7

def aggregate_cross_artifact_features(group):
    """
    Aggregate cross-artifact consistency features.
    """
    ts_change_events = group[group['IsTimestampChange'] == True]
    basic_info_events = group[group['HasBasicInfoChange'] == True]
    close_events = group[group['HasClose'] == True]
    create_events = group[group['HasFileCreate'] == True]
    usnjrnl_events = group[group['is_usnjrnl_event']]
    
    # C.1-4 Event type flags
    has_ts_change = len(ts_change_events) > 0
    has_basic_info = len(basic_info_events) > 0
    has_close = len(close_events) > 0
    has_create = len(create_events) > 0
    
    # C.5 Counts
    num_basic_info = len(basic_info_events)
    num_close = len(close_events)
    num_create = len(create_events)
    
    # C.6 LogFile-USN mismatch (timestamp change but no BASIC_INFO_CHANGE)
    mismatch = has_ts_change and not has_basic_info
    
    # C.7 USN basic detection pattern (Algorithm 5)
    has_basic_pattern = has_basic_info and has_close
    
    # C.8 Total USN events
    num_usnjrnl = len(usnjrnl_events)
    
    return pd.Series({
        'has_logfile_ts_change': has_ts_change,
        'has_usn_basic_info': has_basic_info,
        'has_usn_close': has_close,
        'has_usn_file_create': has_create,
        'num_usn_basic_info': num_basic_info,
        'num_usn_close': num_close,
        'num_usn_file_create': num_create,
        'logfile_usn_mismatch': mismatch,
        'has_usn_basic_pattern': has_basic_pattern,
        'num_usnjrnl_events': num_usnjrnl
    })


print("Cross-artifact feature aggregation function defined.")


Cross-artifact feature aggregation function defined.


In [19]:
# [Cell 9] Temporal Behavior Feature Aggregation (Category D)

def aggregate_temporal_features(group):
    """
    Aggregate temporal behavior features.
    """
    sorted_group = group.sort_values('EventTimestamp')
    timestamps = sorted_group['EventTimestamp'].dropna()
    
    features = {
        'min_inter_event_delta': np.nan,
        'max_inter_event_delta': np.nan,
        'mean_inter_event_delta': np.nan,
        'burstiness_score': 0.0,
        'event_time_span_seconds': np.nan,
        'total_events': len(group)
    }
    
    if len(timestamps) >= 2:
        deltas = timestamps.diff().dropna()
        delta_seconds = deltas.dt.total_seconds()
        
        features['min_inter_event_delta'] = delta_seconds.min()
        features['max_inter_event_delta'] = delta_seconds.max()
        features['mean_inter_event_delta'] = delta_seconds.mean()
        features['event_time_span_seconds'] = (timestamps.max() - timestamps.min()).total_seconds()
        
        # Burstiness: proportion of events within 1 second of each other
        burst_count = (delta_seconds <= 1.0).sum()
        features['burstiness_score'] = burst_count / len(delta_seconds) if len(delta_seconds) > 0 else 0
    
    return pd.Series(features)


print("Temporal feature aggregation function defined.")


Temporal feature aggregation function defined.


In [20]:
# [Cell 10] Detection Flags Computation
# Reference: Oh et al. Core Logic

def compute_detection_flags(file_features):
    """
    Compute logic-based detection flags from file features.
    
    Parameters:
        file_features: DataFrame with aggregated features per file
    
    Returns:
        DataFrame with detection flags
    """
    flags = file_features[['FileFRN', 'dataID', 'FileName', 'FilePath']].copy()
    
    # Flag 1: Backward timestamp jump (Algorithm 3)
    flags['flag_backward_timestamp'] = file_features['num_backward_jumps'] > 0
    
    # Flag 2: Creation time changed (Algorithm 3)
    flags['flag_creation_changed'] = file_features['num_creation_changes'] > 0
    
    # Flag 3: Zero nanoseconds detected (Algorithm 3 additional factor)
    flags['flag_zero_nanoseconds'] = file_features['num_zero_nanosecond_events'] > 0
    
    # Flag 4: LogFile-UsnJrnl mismatch (Algorithms 5-7)
    flags['flag_logfile_usn_mismatch'] = file_features['logfile_usn_mismatch']
    
    # Flag 5: USN basic detection pattern (Algorithm 5)
    flags['flag_usn_basic_pattern'] = file_features['has_usn_basic_pattern']
    
    # Flag 6: Repeated $SI updates (Algorithm 1)
    flags['flag_repeated_si_update'] = file_features['repeated_update_resident_value']
    
    # Flag 7: Only $SI modified
    flags['flag_only_si_modified'] = file_features['only_SI_modified']
    
    # Flag 8: Potential timestomp (combined indicators)
    flags['flag_potential_timestomp'] = (
        (file_features['num_timestamp_changes'] > 0) &
        ((file_features['num_backward_jumps'] > 0) | 
         (file_features['num_zero_nanosecond_events'] > 0))
    )
    
    # Flag 9: Silent timestomp (change without USN evidence)
    flags['flag_silent_timestomp'] = (
        file_features['has_logfile_ts_change'] & 
        ~file_features['has_usn_basic_info']
    )
    
    # Flag 10: High suspicion (multiple indicators)
    flags['flag_high_suspicion'] = (
        flags['flag_backward_timestamp'] & 
        flags['flag_zero_nanoseconds'] & 
        (file_features['num_timestamp_changes'] >= 1)
    )
    
    # Suspicion score (count of true flags)
    flag_cols = [c for c in flags.columns if c.startswith('flag_')]
    flags['suspicion_score'] = flags[flag_cols].sum(axis=1)
    
    return flags


print("Detection flags computation function defined.")


Detection flags computation function defined.


In [21]:
# [Cell 11] Main Feature Extraction Function

def extract_features(dataset_name, input_dir=PHASE2_DIR, output_dir=OUTPUT_DIR):
    """
    Extract file-level features from Phase 2 grouped events.
    
    Parameters:
        dataset_name: Name of the dataset
        input_dir: Phase 2 output directory
        output_dir: Phase 3 output directory
    
    Returns:
        dict with processing statistics
    """
    print(f"\n{'='*60}")
    print(f"FEATURE EXTRACTION: {dataset_name}")
    print(f"{'='*60}")
    
    # Load Phase 2 data
    input_path = input_dir / f"grouped_events_{dataset_name}.csv"
    if not input_path.exists():
        print(f"  ERROR: Input file not found: {input_path}")
        return None
    
    print(f"  Loading {input_path.name}...")
    df = pd.read_csv(input_path, low_memory=False)
    print(f"    Total events: {len(df):,}")
    print(f"    Unique files: {df['FileFRN'].nunique():,}")
    
    # Compute per-event indicators
    print("  Computing per-event indicators...")
    df = compute_event_indicators(df)
    
    # Get file metadata
    print("  Extracting file metadata...")
    file_metadata = df.groupby('FileFRN').agg({
        'dataID': 'first',
        'FileName': 'first',
        'FilePath': 'first'
    }).reset_index()
    
    # Aggregate features by category
    print("  Aggregating timestamp features...")
    ts_features = df.groupby('FileFRN').apply(aggregate_timestamp_features).reset_index()
    
    print("  Aggregating structural features...")
    struct_features = df.groupby('FileFRN').apply(aggregate_structural_features).reset_index()
    
    print("  Aggregating cross-artifact features...")
    cross_features = df.groupby('FileFRN').apply(aggregate_cross_artifact_features).reset_index()
    
    print("  Aggregating temporal features...")
    temporal_features = df.groupby('FileFRN').apply(aggregate_temporal_features).reset_index()
    
    # Merge all features
    print("  Merging feature categories...")
    file_features = file_metadata.merge(ts_features, on='FileFRN', how='left')
    file_features = file_features.merge(struct_features, on='FileFRN', how='left')
    file_features = file_features.merge(cross_features, on='FileFRN', how='left')
    file_features = file_features.merge(temporal_features, on='FileFRN', how='left')
    
    # Fill NaN values
    numeric_cols = file_features.select_dtypes(include=[np.number]).columns
    file_features[numeric_cols] = file_features[numeric_cols].fillna(0)
    
    bool_cols = file_features.select_dtypes(include=[bool]).columns
    file_features[bool_cols] = file_features[bool_cols].fillna(False)
    
    # Compute detection flags
    print("  Computing detection flags...")
    detection_flags = compute_detection_flags(file_features)
    
    # Export features
    features_path = output_dir / f"file_features_{dataset_name}.csv"
    file_features.to_csv(features_path, index=False, encoding='utf-8')
    features_size = features_path.stat().st_size / 1024
    
    # Export flags
    flags_path = output_dir / f"detection_flags_{dataset_name}.csv"
    detection_flags.to_csv(flags_path, index=False, encoding='utf-8')
    flags_size = flags_path.stat().st_size / 1024
    
    # Compute statistics
    stats = {
        'dataset': dataset_name,
        'total_events': len(df),
        'total_files': len(file_features),
        'files_with_ts_changes': (file_features['num_timestamp_changes'] > 0).sum(),
        'files_with_backward_jumps': (file_features['num_backward_jumps'] > 0).sum(),
        'files_with_zero_ns': (file_features['num_zero_nanosecond_events'] > 0).sum(),
        'files_flagged_potential': detection_flags['flag_potential_timestomp'].sum(),
        'files_flagged_high_suspicion': detection_flags['flag_high_suspicion'].sum(),
        'features_size_kb': features_size,
        'flags_size_kb': flags_size
    }
    
    # Print summary
    print(f"\n  --- Summary ---")
    print(f"  Total files: {stats['total_files']:,}")
    print(f"  Files with timestamp changes: {stats['files_with_ts_changes']:,}")
    print(f"  Files with backward jumps: {stats['files_with_backward_jumps']:,}")
    print(f"  Files flagged (potential): {stats['files_flagged_potential']:,}")
    print(f"  Files flagged (high suspicion): {stats['files_flagged_high_suspicion']:,}")
    print(f"  Output: {features_path.name} ({features_size:.1f} KB)")
    print(f"  Output: {flags_path.name} ({flags_size:.1f} KB)")
    
    return stats


print("Main feature extraction function defined.")
print("\nReady to process datasets. Run individual dataset cells below.")


Main feature extraction function defined.

Ready to process datasets. Run individual dataset cells below.


## Training Datasets: PE (01-PE to 12-PE)

Run each cell individually to extract features from one dataset at a time.

In [22]:
# [Cell 13] Extract Features: 01-PE
stats_01PE = extract_features("01-PE")



FEATURE EXTRACTION: 01-PE
  Loading grouped_events_01-PE.csv...
    Total events: 505,582
    Unique files: 37,375
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 37,375
  Files with timestamp changes: 20,430
  Files with backward jumps: 1,109
  Files flagged (potential): 1,111
  Files flagged (high suspicion): 985
  Output: file_features_01-PE.csv (10487.7 KB)
  Output: detection_flags_01-PE.csv (7595.4 KB)


In [23]:
# [Cell 14] Extract Features: 02-PE
stats_02PE = extract_features("02-PE")



FEATURE EXTRACTION: 02-PE
  Loading grouped_events_02-PE.csv...
    Total events: 363,619
    Unique files: 105,251
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 105,251
  Files with timestamp changes: 5,346
  Files with backward jumps: 60
  Files flagged (potential): 60
  Files flagged (high suspicion): 31
  Output: file_features_02-PE.csv (34841.6 KB)
  Output: detection_flags_02-PE.csv (28474.0 KB)


In [24]:
# [Cell 15] Extract Features: 03-PE
stats_03PE = extract_features("03-PE")



FEATURE EXTRACTION: 03-PE
  Loading grouped_events_03-PE.csv...
    Total events: 375,137
    Unique files: 109,985
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 109,985
  Files with timestamp changes: 11,895
  Files with backward jumps: 58
  Files flagged (potential): 58
  Files flagged (high suspicion): 31
  Output: file_features_03-PE.csv (36193.2 KB)
  Output: detection_flags_03-PE.csv (29525.3 KB)


In [25]:
# [Cell 16] Extract Features: 04-PE
stats_04PE = extract_features("04-PE")



FEATURE EXTRACTION: 04-PE
  Loading grouped_events_04-PE.csv...
    Total events: 345,288
    Unique files: 7,882
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 7,882
  Files with timestamp changes: 5,333
  Files with backward jumps: 76
  Files flagged (potential): 76
  Files flagged (high suspicion): 0
  Output: file_features_04-PE.csv (2096.2 KB)
  Output: detection_flags_04-PE.csv (1542.3 KB)


In [26]:
# [Cell 17] Extract Features: 05-PE
stats_05PE = extract_features("05-PE")



FEATURE EXTRACTION: 05-PE
  Loading grouped_events_05-PE.csv...
    Total events: 349,098
    Unique files: 9,451
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 9,451
  Files with timestamp changes: 7,061
  Files with backward jumps: 144
  Files flagged (potential): 144
  Files flagged (high suspicion): 0
  Output: file_features_05-PE.csv (2515.5 KB)
  Output: detection_flags_05-PE.csv (1851.9 KB)


In [27]:
# [Cell 18] Extract Features: 06-PE
stats_06PE = extract_features("06-PE")



FEATURE EXTRACTION: 06-PE
  Loading grouped_events_06-PE.csv...
    Total events: 347,322
    Unique files: 9,434
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 9,434
  Files with timestamp changes: 7,053
  Files with backward jumps: 140
  Files flagged (potential): 140
  Files flagged (high suspicion): 0
  Output: file_features_06-PE.csv (2510.5 KB)
  Output: detection_flags_06-PE.csv (1848.0 KB)


In [28]:
# [Cell 19] Extract Features: 07-PE
stats_07PE = extract_features("07-PE")



FEATURE EXTRACTION: 07-PE
  Loading grouped_events_07-PE.csv...
    Total events: 384,439
    Unique files: 110,336
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 110,336
  Files with timestamp changes: 11,356
  Files with backward jumps: 158
  Files flagged (potential): 158
  Files flagged (high suspicion): 30
  Output: file_features_07-PE.csv (36185.2 KB)
  Output: detection_flags_07-PE.csv (29491.6 KB)


In [29]:
# [Cell 20] Extract Features: 08-PE
stats_08PE = extract_features("08-PE")



FEATURE EXTRACTION: 08-PE
  Loading grouped_events_08-PE.csv...
    Total events: 379,433
    Unique files: 110,541
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 110,541
  Files with timestamp changes: 11,527
  Files with backward jumps: 158
  Files flagged (potential): 158
  Files flagged (high suspicion): 30
  Output: file_features_08-PE.csv (36254.7 KB)
  Output: detection_flags_08-PE.csv (29542.6 KB)


In [30]:
# [Cell 21] Extract Features: 09-PE
stats_09PE = extract_features("09-PE")



FEATURE EXTRACTION: 09-PE
  Loading grouped_events_09-PE.csv...
    Total events: 390,346
    Unique files: 111,889
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 111,889
  Files with timestamp changes: 12,945
  Files with backward jumps: 249
  Files flagged (potential): 250
  Files flagged (high suspicion): 79
  Output: file_features_09-PE.csv (36577.2 KB)
  Output: detection_flags_09-PE.csv (29779.3 KB)


In [31]:
# [Cell 22] Extract Features: 10-PE
stats_10PE = extract_features("10-PE")



FEATURE EXTRACTION: 10-PE
  Loading grouped_events_10-PE.csv...
    Total events: 383,741
    Unique files: 110,574
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 110,574
  Files with timestamp changes: 11,627
  Files with backward jumps: 222
  Files flagged (potential): 223
  Files flagged (high suspicion): 79
  Output: file_features_10-PE.csv (36261.0 KB)
  Output: detection_flags_10-PE.csv (29540.1 KB)


In [32]:
# [Cell 23] Extract Features: 11-PE
stats_11PE = extract_features("11-PE")



FEATURE EXTRACTION: 11-PE
  Loading grouped_events_11-PE.csv...
    Total events: 347,441
    Unique files: 9,415
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 9,415
  Files with timestamp changes: 7,039
  Files with backward jumps: 140
  Files flagged (potential): 140
  Files flagged (high suspicion): 0
  Output: file_features_11-PE.csv (2506.2 KB)
  Output: detection_flags_11-PE.csv (1845.0 KB)


In [33]:
# [Cell 24] Extract Features: 12-PE
stats_12PE = extract_features("12-PE")



FEATURE EXTRACTION: 12-PE
  Loading grouped_events_12-PE.csv...
    Total events: 350,078
    Unique files: 9,160
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 9,160
  Files with timestamp changes: 6,794
  Files with backward jumps: 142
  Files flagged (potential): 143
  Files flagged (high suspicion): 0
  Output: file_features_12-PE.csv (2421.1 KB)
  Output: detection_flags_12-PE.csv (1773.4 KB)


## Training Datasets: APT (10 datasets)

Training APT datasets for model training.


In [34]:
# [Cell 26] Extract Features: 01-APT17
stats_01APT17 = extract_features("01-APT17")



FEATURE EXTRACTION: 01-APT17
  Loading grouped_events_01-APT17.csv...
    Total events: 490,994
    Unique files: 31,455
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 31,455
  Files with timestamp changes: 23,715
  Files with backward jumps: 141
  Files flagged (potential): 142
  Files flagged (high suspicion): 46
  Output: file_features_01-APT17.csv (8158.6 KB)
  Output: detection_flags_01-APT17.csv (5917.1 KB)


In [35]:
# [Cell 27] Extract Features: 03-APT21
stats_03APT21 = extract_features("03-APT21")



FEATURE EXTRACTION: 03-APT21
  Loading grouped_events_03-APT21.csv...
    Total events: 487,612
    Unique files: 28,028
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 28,028
  Files with timestamp changes: 19,988
  Files with backward jumps: 99
  Files flagged (potential): 99
  Files flagged (high suspicion): 48
  Output: file_features_03-APT21.csv (7192.5 KB)
  Output: detection_flags_03-APT21.csv (5160.6 KB)


In [36]:
# [Cell 28] Extract Features: 04-APT28
stats_04APT28 = extract_features("04-APT28")



FEATURE EXTRACTION: 04-APT28
  Loading grouped_events_04-APT28.csv...
    Total events: 494,522
    Unique files: 29,399
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 29,399
  Files with timestamp changes: 21,132
  Files with backward jumps: 117
  Files flagged (potential): 118
  Files flagged (high suspicion): 41
  Output: file_features_04-APT28.csv (7657.4 KB)
  Output: detection_flags_04-APT28.csv (5531.4 KB)


In [37]:
# [Cell 29] Extract Features: 05-APT29
stats_05APT29 = extract_features("05-APT29")



FEATURE EXTRACTION: 05-APT29
  Loading grouped_events_05-APT29.csv...
    Total events: 502,832
    Unique files: 29,860
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 29,860
  Files with timestamp changes: 20,888
  Files with backward jumps: 161
  Files flagged (potential): 162
  Files flagged (high suspicion): 46
  Output: file_features_05-APT29.csv (7758.2 KB)
  Output: detection_flags_05-APT29.csv (5592.5 KB)


In [38]:
# [Cell 30] Extract Features: 06-APT30
stats_06APT30 = extract_features("06-APT30")



FEATURE EXTRACTION: 06-APT30
  Loading grouped_events_06-APT30.csv...
    Total events: 499,102
    Unique files: 22,266
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 22,266
  Files with timestamp changes: 13,406
  Files with backward jumps: 152
  Files flagged (potential): 153
  Files flagged (high suspicion): 41
  Output: file_features_06-APT30.csv (5927.1 KB)
  Output: detection_flags_06-APT30.csv (4210.7 KB)


In [39]:
# [Cell 31] Extract Features: 07-APT37
stats_07APT37 = extract_features("07-APT37")



FEATURE EXTRACTION: 07-APT37
  Loading grouped_events_07-APT37.csv...
    Total events: 491,554
    Unique files: 29,426
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 29,426
  Files with timestamp changes: 21,233
  Files with backward jumps: 157
  Files flagged (potential): 157
  Files flagged (high suspicion): 47
  Output: file_features_07-APT37.csv (7578.2 KB)
  Output: detection_flags_07-APT37.csv (5444.3 KB)


In [40]:
# [Cell 32] Extract Features: 08-APT38
stats_08APT38 = extract_features("08-APT38")



FEATURE EXTRACTION: 08-APT38
  Loading grouped_events_08-APT38.csv...
    Total events: 483,190
    Unique files: 20,478
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 20,478
  Files with timestamp changes: 12,656
  Files with backward jumps: 108
  Files flagged (potential): 108
  Files flagged (high suspicion): 43
  Output: file_features_08-APT38.csv (5369.0 KB)
  Output: detection_flags_08-APT38.csv (3771.3 KB)


In [41]:
# [Cell 33] Extract Features: 10-DarkHotel663
stats_10DarkHotel663 = extract_features("10-DarkHotel663")



FEATURE EXTRACTION: 10-DarkHotel663
  Loading grouped_events_10-DarkHotel663.csv...
    Total events: 453,064
    Unique files: 10,625
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 10,625
  Files with timestamp changes: 8,234
  Files with backward jumps: 74
  Files flagged (potential): 74
  Files flagged (high suspicion): 22
  Output: file_features_10-DarkHotel663.csv (2900.4 KB)
  Output: detection_flags_10-DarkHotel663.csv (2166.1 KB)


In [42]:
# [Cell 34] Extract Features: 11-DarkHotelbbd
stats_11DarkHotelbbd = extract_features("11-DarkHotelbbd")



FEATURE EXTRACTION: 11-DarkHotelbbd
  Loading grouped_events_11-DarkHotelbbd.csv...
    Total events: 452,735
    Unique files: 10,536
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 10,536
  Files with timestamp changes: 8,121
  Files with backward jumps: 70
  Files flagged (potential): 70
  Files flagged (high suspicion): 23
  Output: file_features_11-DarkHotelbbd.csv (2871.2 KB)
  Output: detection_flags_11-DarkHotelbbd.csv (2145.0 KB)


In [43]:
# [Cell 35] Extract Features: 14-Winnti53b
stats_14Winnti53b = extract_features("14-Winnti53b")



FEATURE EXTRACTION: 14-Winnti53b
  Loading grouped_events_14-Winnti53b.csv...
    Total events: 424,926
    Unique files: 16,703
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 16,703
  Files with timestamp changes: 15,096
  Files with backward jumps: 1,017
  Files flagged (potential): 1,017
  Files flagged (high suspicion): 967
  Output: file_features_14-Winnti53b.csv (4434.9 KB)
  Output: detection_flags_14-Winnti53b.csv (3286.3 KB)


## Validation Datasets (5 datasets)

These datasets are processed independently for model evaluation.
They will NOT be included in the training data merge.


In [44]:
# [Cell 37] Extract Features: 02-APT19 (Validation)
stats_02APT19 = extract_features("02-APT19")


FEATURE EXTRACTION: 02-APT19
  Loading grouped_events_02-APT19.csv...
    Total events: 475,043
    Unique files: 21,336
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 21,336
  Files with timestamp changes: 12,600
  Files with backward jumps: 135
  Files flagged (potential): 136
  Files flagged (high suspicion): 39
  Output: file_features_02-APT19.csv (5698.0 KB)
  Output: detection_flags_02-APT19.csv (4064.6 KB)


In [45]:
# [Cell 38] Extract Features: 09-APT40 (Validation)
stats_09APT40 = extract_features("09-APT40")



FEATURE EXTRACTION: 09-APT40
  Loading grouped_events_09-APT40.csv...
    Total events: 497,444
    Unique files: 21,672
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 21,672
  Files with timestamp changes: 13,527
  Files with backward jumps: 125
  Files flagged (potential): 126
  Files flagged (high suspicion): 45
  Output: file_features_09-APT40.csv (5775.6 KB)
  Output: detection_flags_09-APT40.csv (4098.6 KB)


In [46]:
# [Cell 39] Extract Features: 12-Kimsuky (Validation)
stats_12Kimsuky = extract_features("12-Kimsuky")



FEATURE EXTRACTION: 12-Kimsuky
  Loading grouped_events_12-Kimsuky.csv...
    Total events: 454,971
    Unique files: 14,869
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 14,869
  Files with timestamp changes: 12,550
  Files with backward jumps: 73
  Files flagged (potential): 73
  Files flagged (high suspicion): 20
  Output: file_features_12-Kimsuky.csv (3876.5 KB)
  Output: detection_flags_12-Kimsuky.csv (2886.4 KB)


In [47]:
# [Cell 40] Extract Features: 13-Winnti731 (Validation)
stats_13Winnti731 = extract_features("13-Winnti731")



FEATURE EXTRACTION: 13-Winnti731
  Loading grouped_events_13-Winnti731.csv...
    Total events: 401,719
    Unique files: 16,741
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 16,741
  Files with timestamp changes: 14,524
  Files with backward jumps: 71
  Files flagged (potential): 71
  Files flagged (high suspicion): 22
  Output: file_features_13-Winnti731.csv (4364.1 KB)
  Output: detection_flags_13-Winnti731.csv (3265.7 KB)


In [48]:
# [Cell 41] Extract Features: LoneWolf (Validation)
stats_LoneWolf = extract_features("LoneWolf")



FEATURE EXTRACTION: LoneWolf
  Loading grouped_events_LoneWolf.csv...
    Total events: 481,400
    Unique files: 17,897
  Computing per-event indicators...
  Extracting file metadata...
  Aggregating timestamp features...
  Aggregating structural features...
  Aggregating cross-artifact features...
  Aggregating temporal features...
  Merging feature categories...
  Computing detection flags...

  --- Summary ---
  Total files: 17,897
  Files with timestamp changes: 1,559
  Files with backward jumps: 309
  Files flagged (potential): 313
  Files flagged (high suspicion): 304
  Output: file_features_LoneWolf.csv (5228.8 KB)
  Output: detection_flags_LoneWolf.csv (3618.5 KB)


## Merge Training Datasets

Combine all 22 training feature files into single CSVs for Phase 4 model training:
- `file_features_training.csv`
- `detection_flags_training.csv`


In [49]:
# [Cell 43] Merge Training Features

print("=" * 60)
print("MERGING TRAINING FEATURES")
print("=" * 60)

training_datasets = TRAINING_PE + TRAINING_APT
print(f"\nTraining datasets to merge: {len(training_datasets)}")

# Merge file features
features_dfs = []
flags_dfs = []

for dataset in training_datasets:
    features_path = OUTPUT_DIR / f"file_features_{dataset}.csv"
    flags_path = OUTPUT_DIR / f"detection_flags_{dataset}.csv"
    
    if features_path.exists() and flags_path.exists():
        print(f"  Loading {dataset}...")
        features_dfs.append(pd.read_csv(features_path, low_memory=False))
        flags_dfs.append(pd.read_csv(flags_path, low_memory=False))
    else:
        print(f"  WARNING: {dataset} files not found, skipping...")

if features_dfs:
    # Concatenate features
    print("\n  Merging file features...")
    df_features_training = pd.concat(features_dfs, ignore_index=True)
    
    features_output = OUTPUT_DIR / "file_features_training.csv"
    df_features_training.to_csv(features_output, index=False, encoding='utf-8')
    features_size_mb = features_output.stat().st_size / (1024 * 1024)
    
    print(f"  Total files: {len(df_features_training):,}")
    print(f"  Datasets: {df_features_training['dataID'].nunique()}")
    print(f"  Output: {features_output.name} ({features_size_mb:.2f} MB)")
    
    # Concatenate flags
    print("\n  Merging detection flags...")
    df_flags_training = pd.concat(flags_dfs, ignore_index=True)
    
    flags_output = OUTPUT_DIR / "detection_flags_training.csv"
    df_flags_training.to_csv(flags_output, index=False, encoding='utf-8')
    flags_size_mb = flags_output.stat().st_size / (1024 * 1024)
    
    print(f"  Total files: {len(df_flags_training):,}")
    print(f"  Output: {flags_output.name} ({flags_size_mb:.2f} MB)")
    
    # Summary statistics
    print(f"\n--- Training Data Summary ---")
    print(f"Files flagged (potential_timestomp): {df_flags_training['flag_potential_timestomp'].sum():,}")
    print(f"Files flagged (high_suspicion): {df_flags_training['flag_high_suspicion'].sum():,}")
else:
    print("\nERROR: No training datasets found to merge.")


MERGING TRAINING FEATURES

Training datasets to merge: 22
  Loading 01-PE...
  Loading 02-PE...
  Loading 03-PE...
  Loading 04-PE...
  Loading 05-PE...
  Loading 06-PE...
  Loading 07-PE...
  Loading 08-PE...
  Loading 09-PE...
  Loading 10-PE...
  Loading 11-PE...
  Loading 12-PE...
  Loading 01-APT17...
  Loading 03-APT21...
  Loading 04-APT28...
  Loading 05-APT29...
  Loading 06-APT30...
  Loading 07-APT37...
  Loading 08-APT38...
  Loading 10-DarkHotel663...
  Loading 11-DarkHotelbbd...

  Merging file features...
  Total files: 953,366
  Datasets: 21
  Output: file_features_training.csv (287.22 MB)

  Merging detection flags...
  Total files: 953,366
  Output: detection_flags_training.csv (227.29 MB)

--- Training Data Summary ---
Files flagged (potential_timestomp): 3,744
Files flagged (high_suspicion): 1,622


In [50]:
# [Cell 44] Generate Summary Report

print("=" * 60)
print("PHASE 3 PROCESSING SUMMARY")
print("=" * 60)

# Collect all stats
all_stats = []
for var_name in dir():
    if var_name.startswith('stats_') and isinstance(eval(var_name), dict):
        all_stats.append(eval(var_name))

if all_stats:
    df_summary = pd.DataFrame(all_stats)
    
    print(f"\nDatasets processed: {len(df_summary)}")
    print(f"\n{'Dataset':<20} {'Files':>10} {'TS Chg':>10} {'Backward':>10} {'Flagged':>10}")
    print("-" * 65)
    
    for _, row in df_summary.iterrows():
        print(f"{row['dataset']:<20} {row['total_files']:>10,} {row['files_with_ts_changes']:>10,} {row['files_with_backward_jumps']:>10,} {row['files_flagged_potential']:>10,}")
    
    print("-" * 65)
    print(f"{'TOTAL':<20} {df_summary['total_files'].sum():>10,} {df_summary['files_with_ts_changes'].sum():>10,} {df_summary['files_with_backward_jumps'].sum():>10,} {df_summary['files_flagged_potential'].sum():>10,}")
    
    # Save summary
    summary_path = OUTPUT_DIR / "phase3_processing_summary.csv"
    df_summary.to_csv(summary_path, index=False)
    print(f"\nSummary saved to: {summary_path}")
else:
    print("\nNo processing statistics available.")

print("\n" + "=" * 60)
print("PHASE 3 COMPLETE")
print("=" * 60)
print("\nNext Step: Phase 4 - Model Training & Evaluation")


PHASE 3 PROCESSING SUMMARY

Datasets processed: 27

Dataset                   Files     TS Chg   Backward    Flagged
-----------------------------------------------------------------
01-APT17                 31,455     23,715        141        142
01-PE                    37,375     20,430      1,109      1,111
02-APT19                 21,336     12,600        135        136
02-PE                   105,251      5,346         60         60
03-APT21                 28,028     19,988         99         99
03-PE                   109,985     11,895         58         58
04-APT28                 29,399     21,132        117        118
04-PE                     7,882      5,333         76         76
05-APT29                 29,860     20,888        161        162
05-PE                     9,451      7,061        144        144
06-APT30                 22,266     13,406        152        153
06-PE                     9,434      7,053        140        140
07-APT37                 29,426     2

## Output File Reference

### Training Data (Combined)
- `file_features_training.csv` - All 22 training datasets merged
- `detection_flags_training.csv` - Detection flags for training

### Training Datasets (Individual)
- `file_features_[dataset].csv` - Per-dataset features
- `detection_flags_[dataset].csv` - Per-dataset flags

### Validation Datasets (Separate)
- `file_features_02-APT19.csv`, `detection_flags_02-APT19.csv`
- `file_features_09-APT40.csv`, `detection_flags_09-APT40.csv`
- `file_features_12-Kimsuky.csv`, `detection_flags_12-Kimsuky.csv`
- `file_features_13-Winnti731.csv`, `detection_flags_13-Winnti731.csv`
- `file_features_LoneWolf.csv`, `detection_flags_LoneWolf.csv`

### Processing Summary
- `phase3_processing_summary.csv` - Statistics for all datasets
